<a href="https://colab.research.google.com/github/JasonPeer/inventory_volatility_analysis/blob/main/notebooks/02_demand_volatility_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
#Import Environment
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('/content/drive/MyDrive/YBF_PP/ca1_long_sales_small.csv')

df.shape

#Check the structure
df.head()

#Aggregating the demand distribution into each SKU
sku_summary = df.groupby("item_id")["demand"].agg(
    mean_demand="mean",
    std_demand="std",
    median_demand="median",
    total_demand="sum"
).reset_index()

sku_summary.head()

#Calculate the CV
sku_summary["cv"] = sku_summary["std_demand"] / sku_summary["mean_demand"]

#Prevent that mean = 0
sku_summary.loc[sku_summary["mean_demand"] == 0, "cv"] = np.nan

sku_summary.head()

#Check the statistic distribution
sku_summary[["mean_demand", "std_demand", "cv", "total_demand"]].describe()
  #We find out that CA_1 has totally 3049 SKUs.

  #Also,CA_1 has a mean number of demand around 1.32, while 25% of SKU only sell 0.26 unit per day, 50% SKU only sell 0.59 unit per day.
  #That indicates that a few goods are very pourpular, while most of the other goods are sold very slow.

  #std_demand is around 1.68, even higher than mean demand. That may indicate that demand volatility is really obvious.
  #Max of 57.86 means a few SKU has both high demand and high volatility.

  #The CV data is very beautiful(I'm joking) because a mean of 2.10 and medium of 1.84 is much higher than 1, Max is even > 10.
  #That indicates that the volatility of these goods are really really strong. I like it!
  #A fixed reorder point strategy may has a higher posibility fail to handle this strong volatility.

  #Total demand looks extremly unbalanced. We can tell that this is a typical long trail retail.

  #All in all, CA_1 store contains 3,049 SKUs with highly heterogeneous demand patterns.
  #Median daily demand is below one unit per day, while demand volatility (CV) averages above 2, indicating substantial variability across items.

#Draw the CV distribution picture.
# plt.figure(figsize=(8,5))
# plt.hist(sku_summary["cv"].dropna(), bins=50)
# plt.xlabel("Coefficient of Variation (CV)")
# plt.ylabel("Number of SKUs")
# plt.title("Distribution of Demand Volatility Across SKUs")
# plt.show()

#Great Distribution! Now we segment the CV data
def classify_cv(cv):
    if cv < 1:
        return "Low"
    elif cv < 2:
        return "Medium"
    else:
        return "High"

sku_summary["cv_group"] = sku_summary["cv"].apply(classify_cv)

sku_summary["cv_group"].value_counts()

sku_summary.to_csv('/content/drive/MyDrive/YBF_PP/ca1_sku_volatility_groups.csv', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
